# Test: inspect a packaged NWB

Loads an NWB produced by this capsule and checks the events / intervals
tables, HED tags, and task parameters. Run `code/run` first, then point
`NWB_PATH` at a file in `../results/` (done automatically below).

In [ ]:
import glob, os
from pynwb import NWBHDF5IO
import ndx_events               # noqa: F401  registers the EventsTable type
import ndx_hed                  # noqa: F401  registers HED types
import ndx_change_detection_task  # noqa: F401  registers task-parameter type

candidates = sorted(glob.glob('../results/*.nwb')) or sorted(glob.glob('/results/*.nwb'))
assert candidates, 'No NWB found in ../results or /results — run code/run first.'
NWB_PATH = candidates[0]
print('Loading', NWB_PATH, f'({os.path.getsize(NWB_PATH)/1e6:.1f} MB)')
io = NWBHDF5IO(NWB_PATH, mode='r', load_namespaces=True)
nwb = io.read()

## Events table (point events)

The `ndx-events` `EventsTable` lives at `nwb.events['events']`.

In [ ]:
events = nwb.events['events']
df = events[:]
print('rows:', len(df), '| columns:', list(df.columns))
print('\nevent_type counts:')
print(df['event_type'].value_counts())
df.head(10)

## HED tags

For the EventsTable, each categorical column carries a `MeaningsTable`
whose `meaning` column holds the base HED string. The interval tables
(e.g. `stimulus_presentations`) instead carry a direct `HED` column.

In [ ]:
for col in ['event_type', 'lick_classification', 'reward_type', 'lick_bouts']:
    meanings = getattr(events[col], 'meanings', None)
    if meanings is None:
        continue
    print(f'=== {col} HED meanings ===')
    mdf = meanings[:]
    for _, r in mdf.iterrows():
        print(f"  {r['value']:>14}  ->  {r['meaning']}")
    print()

## Intervals: stimulus presentations / trials / movie / epochs

In [ ]:
for name, tbl in nwb.intervals.items():
    print(f'{name}: {len(tbl)} rows | cols = {list(tbl.colnames)}')

sp = nwb.intervals['stimulus_presentations'][:]
print('\nstimulus_presentations sample (with direct HED column):')
sp[['start_time', 'stop_time', 'image_name', 'is_change', 'omitted', 'HED']].head(6)

## Task parameters (ndx-change-detection-task extension)

In [ ]:
tp = nwb.lab_meta_data.get('task_parameters')
if tp is not None:
    for f in sorted(tp.fields):
        print(f'{f:>28} = {getattr(tp, f, None)}')
else:
    print('No task_parameters lab_meta_data found.')

In [ ]:
io.close()
print('OK — NWB loaded and inspected cleanly.')